In [0]:
        %sql

    WITH pct_raw AS (            -- one row per percentage statement
        SELECT
            psr.canonical_id AS ownership_id,
            pbs.dataset,
            replace(pbs.value, '%', '') AS pct_text,
            try_cast(split_part(pct_text, '–', 1)  AS DOUBLE) AS lo,
            try_cast(split_part(pct_text, '–', -1) AS DOUBLE) AS hi
        FROM pep_dev.bronze.statements AS pbs
        JOIN pep_dev.silver.referent AS psr 
        ON psr.entity_id = pbs.entity_id
        WHERE pbs.schema = 'Ownership'
        AND pbs.prop = 'percentage'
        AND pbs._snapshot_version = (SELECT MAX(snapshot_version) FROM pep_dev.bronze.snapshot_version_control)
    ),
    pct_scaled AS (              -- c4ads writes fractions
        SELECT ownership_id,
            CASE WHEN dataset = 'c4ads_xinjiang' THEN lo * 100 ELSE lo END AS lo,
            CASE WHEN dataset = 'c4ads_xinjiang' THEN hi * 100 ELSE hi END AS hi
        FROM pct_raw
    ),
    pct AS (                     -- one row per record; 0 and >100 become NULL
        SELECT ownership_id,
            round(MIN(CASE WHEN lo > 0 AND lo <= 100 THEN lo END), 4) AS pct_min,
            round(MAX(CASE WHEN hi > 0 AND hi <= 100 THEN hi END), 4) AS pct_max
        FROM pct_scaled
        GROUP BY ownership_id
    )


    ,pivot AS (                   -- one row per ownership record
        SELECT
            psr.canonical_id AS ownership_id,
            MAX(pbs.value) FILTER (WHERE pbs.prop = 'owner') AS owner_ref,
            MAX(pbs.value) FILTER (WHERE pbs.prop = 'asset') AS asset_ref,
            count(DISTINCT pbs.value) FILTER (WHERE pbs.prop = 'owner') AS n_owner,  -- grain check
            count(DISTINCT pbs.value) FILTER (WHERE pbs.prop = 'asset') AS n_asset,  -- grain check
            try_cast(MIN(pbs.value) FILTER (WHERE pbs.prop = 'startDate') AS DATE) AS start_date,
            try_cast(MAX(pbs.value) FILTER (WHERE pbs.prop = 'endDate')   AS DATE) AS end_date,
            collect_set(pbs.dataset) AS datasets,
            MIN(pbs._snapshot_version) AS _first_seen_snapshot,
            MAX(pbs._snapshot_version) AS _last_seen_snapshot
        FROM pep_dev.bronze.statements AS pbs
        JOIN pep_dev.silver.referent AS psr
        ON psr.entity_id = pbs.entity_id
        WHERE pbs.schema = 'Ownership'
        AND pbs._snapshot_version = (SELECT MAX(snapshot_version) FROM pep_dev.bronze.snapshot_version_control)
        GROUP BY psr.canonical_id
    ),

    kind as (
        SELECT 
        canonical_id,
    ( CASE
            WHEN array_contains(collect_set(referent_kind), 'Person') 
                THEN 'Person'
            WHEN array_contains(collect_set(referent_kind), 'Company')
                THEN 'Company'
            WHEN array_contains(collect_set(referent_kind), 'Organization')  
                THEN 'Organization'
            WHEN array_contains(collect_set(referent_kind), 'LegalEntity') 
                THEN 'LegalEntity' 
            ELSE element_at(collect_set(referent_kind), 1)
                END  ) as node_type

            
            

        FROM pep_dev.silver.referent
        GROUP BY canonical_id
    )


    , resolved as (
        SELECT p.*,
                coalesce(psr.canonical_id,p.owner_ref) as owner_id, -- get canonical id as it covers multiple entity_id inter-linked or  owner_ref if  canonical_id is null hence owner is found
                coalesce(psr2.canonical_id,p.asset_ref) as asset_id  -- get canonical id as it covers multiple entity_id inter-linked or  asset_ref if  canonical_id is null hence asset is found
        FROM pivot as p
        LEFT JOIN pep_dev.silver.referent as psr
        ON  p.owner_ref=psr.entity_id
        LEFT JOIN pep_dev.silver.referent as psr2
        ON  p.asset_ref=psr2.entity_id
        WHERE p.owner_ref IS NOT NULL
        AND p.asset_ref IS NOT NULL

    ),
    final as (
        SELECT 

        r.ownership_id,
        r.owner_ref,
        r.asset_ref,
        r.owner_id,
        r.asset_id,
        ka.node_type as owner_type,
        ko.node_type as asset_type,
        p.pct_min,
        p.pct_max,
        r.start_date,
        r.end_date,
        coalesce(r.end_date>=to_date(
            left(r._last_seen_snapshot,8), 'yyyyMMdd'
        ), true ) AS is_current,
        (r.owner_id=r.asset_id) AS is_self_owning,
        r.datasets,
        r._last_seen_snapshot,
        r._first_seen_snapshot


        FROM resolved as r
        LEFT JOIN pct as p
        ON  r.ownership_id=p.ownership_id
        LEFT JOIN kind as ka
        ON r.owner_id=ka.canonical_id -- to know owner node kind('person','company',...)
        LEFT JOIN kind as ko
        ON r.asset_id=ko.canonical_id -- to know owner node kind('person','company',...)
        -- why two l joins to fetch owner and its asset kind in one record/row 
            
    )



MERGE INTO pep_dev.silver.ownership AS t
USING final AS s 
ON t.ownership_id = s.ownership_id
WHEN MATCHED THEN UPDATE SET 
    t.owner_ref = s.owner_ref,
    t.owner_id = s.owner_id,
    t.owner_type = s.owner_type,
    t.asset_ref = s.asset_ref,
    t.asset_id = s.asset_id,
    t.asset_type = s.asset_type,
    t.pct_min = s.pct_min,
    t.pct_max = s.pct_max,
    t.start_date = s.start_date,
    t.end_date = s.end_date,
    t.is_current = s.is_current,
    t.is_self_owning = s.is_self_owning,
    t.datasets = s.datasets,
    t._last_seen_snapshot = s._last_seen_snapshot
WHEN NOT MATCHED THEN INSERT 
(
    ownership_id,
    owner_ref,
    owner_id,
    owner_type,
    asset_ref,
    asset_id,
    asset_type,
    pct_min,
    pct_max,
    start_date,
    end_date,
    is_current,
    is_self_owning,
    datasets,
    _last_seen_snapshot,
    _first_seen_snapshot

) 
values 
(
    s.ownership_id,
    s.owner_ref,
    s.owner_id,
    s.owner_type,
    s.asset_ref,
    s.asset_id,
    s.asset_type,
    s.pct_min,
    s.pct_max,
    s.start_date,
    s.end_date,
    s.is_current,
    s.is_self_owning,
    s.datasets,
    s._last_seen_snapshot,
    s._first_seen_snapshot
)
-- DBTITLE 1



In [0]:
%sql
-- SELECT count(*), count(DISTINCT ownership_id) FROM pep_dev.silver.ownership

In [0]:
# %sql
# SELECT o.asset_type, count(*) AS n, min(o.asset_id) AS example_id
# FROM pep_dev.silver.ownership AS o
# LEFT ANTI JOIN pep_dev.silver.legal_entity AS le
#   ON le.canonical_id = o.asset_id
# WHERE o.asset_type NOT IN ('Person', 'Vessel', 'Airplane')
# GROUP BY o.asset_type